# PostgreSQL for data pipelines

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/postgres/postgres.ipynb)

PostgreSQL combines relational constraints, transactions and a broad SQL implementation. For pipelines, understand business keys, statement-level versus transaction-level snapshots, upserts and change capture.

In [ ]:
# Setup: installs this notebook's packages when they are missing (e.g. in Google Colab).
import importlib.util
import subprocess
import sys

PACKAGES = {'duckdb': 'duckdb>=1.0', 'dbt': 'dbt-core>=1.9,<2', 'dbt.adapters.duckdb': 'dbt-duckdb>=1.9,<2'}


def _missing(module: str, spec: str) -> bool:
    """True when the module is absent or its installed version does not satisfy spec."""
    try:
        if importlib.util.find_spec(module) is None:
            return True
    except ModuleNotFoundError:
        return True
    try:
        from importlib.metadata import PackageNotFoundError, version

        from packaging.requirements import Requirement
    except ImportError:
        return False
    requirement = Requirement(spec)
    try:
        return not requirement.specifier.contains(version(requirement.name), prereleases=True)
    except PackageNotFoundError:
        return False


missing = [spec for module, spec in PACKAGES.items() if _missing(module, spec)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("ready:", ", ".join(PACKAGES))

## Decisions and trade-offs

Use a unique constraint on the conflict key before relying on ON CONFLICT. Avoid updating unchanged rows just to advance a pipeline: writes generate WAL and can increase replica and CDC load. A transaction on one connection does not provide exactly-once delivery across two services.

Row-level security defaults to deny after it is enabled without an applicable policy. Owners normally bypass RLS; superusers and BYPASSRLS roles do too. Test policies using the actual application role.

```sql
INSERT INTO customers(id, email) VALUES (1, 'ana@example.com')
ON CONFLICT (id) DO UPDATE SET email = EXCLUDED.email
WHERE customers.email IS DISTINCT FROM EXCLUDED.email;
```

The executable exercise below uses DuckDB to demonstrate transaction rollback, not PostgreSQL server behavior. No PostgreSQL instance is started.

[PostgreSQL row security](https://www.postgresql.org/docs/current/ddl-rowsecurity.html)

## Worked example

Run the small fixture and inspect each assertion. Change one input to see which contract fails.

In [ ]:
import duckdb
with duckdb.connect() as con:
    con.sql('create table accounts(id integer primary key, balance integer check(balance>=0))')
    con.sql('insert into accounts values (1,100),(2,50)')
    con.sql('begin transaction')
    con.sql('update accounts set balance=balance-20 where id=1')
    con.sql('rollback')
    assert con.sql('select sum(balance) from accounts').fetchall()[0]==(150,)
    assert con.sql('select balance from accounts where id=1').fetchall()[0]==(100,)

## Practice

Add a failure case, explain the expected outcome before running it, and identify which guarantee would need a real service or a larger dataset to verify.